# 🧠 RAG Foundations Masterclass: From Keyword Match to Two-Stage Retrieval
**Core 1: Theoretical Depth & System Fundamentals**

This notebook deconstructs the architecture of Retrieval-Augmented Generation (RAG). It traces the evolution of a search system from deterministic lexical matching to a probabilistic, two-stage vector retrieval pipeline utilizing semantic chunking, dimensionality reduction, and cross-encoder reranking.

**Architectural Highlights:**
* **Lexical vs. Semantic:** Contrasting naive keyword mapping with high-dimensional vector embeddings.
* **Semantic Chunking:** Utilizing LLMs and Pydantic for structured, context-aware document splitting.
* **Vector Math & Visualization:** Projecting 384-dimensional embeddings into 2D/3D space using t-SNE.
* **Two-Stage Pipeline:** Bi-encoder retrieval (`BAAI/bge-small-en-v1.5`) paired with local cross-encoder reranking (`ms-marco-MiniLM-L6-v2`) for cost-optimized precision.

In [1]:
# Environments and Imports
import os
import time
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

# Machine Learning & NLP
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Vector DB & LLM
import chromadb
from openai import OpenAI, RateLimitError
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# Rate limiting 
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

# TOGGLE: Set to False if you want to force re-processing from the LLM
LOAD_FROM_DISK = True
# LOAD_FROM_DISK = False

# Load API keys(Gemini used for generation/chunking only; embeddings/reranking are local)
load_dotenv(override=True)
gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
MODEL = "gemini-3.5-flash-lite"

KNOWLEDGE_BASE_PATH = Path("../data/knowledge-base")
DB_NAME = "vector_db"
CHUNKS_CACHE_FILE = "processed_chunks.json"

### 1. The Baseline: Lexical Keyword Matching (Why RAG Needs Vectors)
Early search systems relied on exact string matching. While fast and computationally cheap, lexical search fails to capture user intent or semantic synonyms.

In [2]:
# Naive dictionary-based knowledge store
knowledge = {}
if KNOWLEDGE_BASE_PATH.exists():
    for file in KNOWLEDGE_BASE_PATH.rglob("*.md"):
        name = file.stem.split(' ')[-1].lower()
        with open(file, "r", encoding="utf-8") as f:
            knowledge[name] = f.read()

def get_lexical_context(message):
    """Naive keyword matching algorithm(using only title matching)."""
    words = ''.join(ch for ch in message if ch.isalpha() or ch.isspace()).lower().split()
    return [knowledge[word] for word in words if word in knowledge]

# Fails on synonyms or slight variations (e.g., "Alex" instead of "Lancaster")
print("Lexical Match for 'Alex':", len(get_lexical_context("Who is Alex?")))
print("Lexical Match for 'Lancaster':", len(get_lexical_context("Who is Lancaster?")))

Lexical Match for 'Alex': 0
Lexical Match for 'Lancaster': 1


### 2. Advanced Document Processing: LLM-Assisted Semantic Chunking
Instead of arbitrary character-count splitting, we use an LLM with structured outputs (Pydantic) to generate context-aware chunks. Each chunk includes a generated headline and summary to maximize retrieval surface area.

(Rate Limited & Cached)

In [3]:
# Pydantic Chunking Refractored
class Chunk(BaseModel):
    headline: str = Field(description="Brief heading likely to surface in queries")
    summary: str = Field(description="Summary answering common questions")
    original_text: str = Field(description="Exact original text snippet")

class DocumentChunks(BaseModel):
    chunks: list[Chunk]

def load_documents():
    docs = []
    if not KNOWLEDGE_BASE_PATH.exists():
        print(f"Warning: Directory {KNOWLEDGE_BASE_PATH} does not exist. Skipping doc load.")
        return docs

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        if folder.is_dir():
            for file in folder.rglob("*.md"):
                with open(file, "r", encoding="utf-8") as f:
                    docs.append({"type": folder.name, "source": file.as_posix(), "text": f.read()})
    return docs

# Retry Decorator for Exponential backoff for rate limits
@retry(
    wait=wait_random_exponential(min=2, max=60), # wait between 2 and 60 seconds
    stop=stop_after_attempt(6), # giveup after 6 tries
    retry=retry_if_exception_type(RateLimitError) # only retry on Rate Limit errors
)
def semantic_chunk_document(document, avg_chunk_size=500):
    """Generates structured chunks using Gemini's native JSON parsing."""
    how_many = max(1, len(document["text"]) // avg_chunk_size)
    prompt = f"""
    Split this {document['type']} document from {document['source']} into ~{how_many} overlapping chunks.
    Provide a headline, summary, and the original text for each.
    \n\n{document['text']}
    """
    response = gemini.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=DocumentChunks
    )
    parsed_chunks = response.choices[0].message.parsed.chunks

    # Inject metadata into each chunk before returning(converting to standard dictionary to safely inject metadata)
    enriched_chunks = []
    for c in parsed_chunks:
        chunk_dict = c.model_dump()
        chunk_dict["type"] = document['type']
        chunk_dict["source"] = document['source']
        enriched_chunks.append(chunk_dict)
    return parsed_chunks

In [4]:
# OPTIONAL RESTORATION/GENERATION OF CHUNKS
restored_chunks = []

if LOAD_FROM_DISK and os.path.exists(CHUNKS_CACHE_FILE):
    print(f"Loading cached chunks from {CHUNKS_CACHE_FILE}...")
    with open(CHUNKS_CACHE_FILE, "r", encoding="utf-8") as f:
        restored_chunks = json.load(f)
    print(f"Restored {len(restored_chunks)} chunks from disk.")

else:
    print("Processing documents via API...")
    documents = load_documents()

    for doc in tqdm(documents, desc="Chunking Documents"):
        try:
            doc_chunks = semantic_chunk_document(doc)
            restored_chunks.extend(doc_chunks)
        except Exception as e:
            print(f"\nFailed to process {doc['source']} after multiple retries. Error: {e}")

    # Save to disk for next time
    with open(CHUNKS_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump([c.model_dump() for c in restored_chunks], f, indent=2)
    print(f"Saved {len(restored_chunks)} chunks to {CHUNKS_CACHE_FILE}.")

Loading cached chunks from processed_chunks.json...
Restored 544 chunks from disk.


In [5]:
# Correcting the chunking error to include type and source fields
print(f"Loading chunks from {CHUNKS_CACHE_FILE}...")
with open(CHUNKS_CACHE_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Load the original documents from disk
documents = load_documents()

print(f"Recovering missing metadata for {len(chunks)} chunks...")
recovered_count = 0

for chunk in chunks:
    chunk_text = chunk.get("original_text", "")

    # Set defaults just in case
    chunk["type"] = "Unknown"
    chunk["source"] = "Unknown"

    # Search for this chunk's text inside the original Markdown files
    for doc in documents:
        if chunk_text in doc["text"]:
            chunk["type"] = doc["type"]
            chunk["source"] = doc["source"]
            recovered_count += 1
            break # Found the matching file, move to the next chunk
        
# Overwrite the broken JSON file with the newly fixed data
with open(CHUNKS_CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2)

print(f"Success! Recovered metadata for {recovered_count}/{len(chunks)} chunks.")
print("Saved fixed data to processed_chunks.json. No API calls were made.")

Loading chunks from processed_chunks.json...
Recovering missing metadata for 544 chunks...
Success! Recovered metadata for 458/544 chunks.
Saved fixed data to processed_chunks.json. No API calls were made.


### 3. Vector Embeddings & Dimensionality Reduction (Math & Visualization)
We translate text into mathematical representations using `BAAI/bge-small-en-v1.5`. We then use t-SNE (t-Distributed Stochastic Neighbor Embedding) to project these high-dimensional vectors (384 dimensions) into 2D and 3D space to visualize semantic clustering.

In [6]:
# Initialize the local bi-encoder
print("Loading BAAI/bge-large-en-v1.5...")
encoder = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Setup ChromaDB
chroma_client = chromadb.PersistentClient(path=DB_NAME)
collection = chroma_client.get_or_create_collection("enterprise_docs_large")

# Ingest if DB is empty 
if collection.count() == 0 and restored_chunks:
    print("\nVector database empty. Ingesting chunks...")

    # Extract the text and metadata from Chunk onjects
    texts = [chunk["original_text"] for chunk in restored_chunks]
    metadatas = [
        {
            "type": chunk["type"], 
            "headline": chunk["headline"],
            "summary": chunk["summary"],
            "source": chunk["source"]
        } 
        for chunk in restored_chunks
    ]

    # ChromaDB requires a unique ID for every single chunk
    ids = [f"chunk_{i}" for i in range(len(restored_chunks))]

    print("Generating embeddings...")
    embeddings = encoder.encode(texts, show_progress_bar=True).tolist()

    print("Saving to ChromaDB...")
    collection.upsert(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas
    )
    print("Ingestion complete!")

Loading BAAI/bge-large-en-v1.5...


In [7]:
# Visualization
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

# Safety check if embeddings exist and are not empty
if result.get('embeddings') is not None and len(result['embeddings']) > 0:
    print("\nGenerating 2D and 3D t-SNE Plot...")
    vectors = np.array(result['embeddings'])

    # Extract metadata for grouping (colors) and hover text
    doc_types = np.array([meta.get('type', 'Unknown') for meta in result['metadatas']])
    headlines = np.array([meta.get('headline', 'No Headline') for meta in result['metadatas']])
    unique_types = np.unique(doc_types)

    # 2D plot
    tsne_2d = TSNE(n_components=2, random_state=42)
    reduced_vectors_2d = tsne_2d.fit_transform(vectors)

    fig_2d = go.Figure()
    for doc_type in unique_types:
        idx = doc_types == doc_type
        fig_2d.add_trace(go.Scatter(
            x=reduced_vectors_2d[idx, 0], 
            y=reduced_vectors_2d[idx, 1],
            mode='markers', 
            name=doc_type,             # Adds legend entry
            text=headlines[idx],        # Custom text for hover
            hoverinfo='text+name',      # Show headline and category on hover
            marker=dict(size=8, opacity=0.7)
    ))
    fig_2d.update_layout(title='Semantic Clustering: 2D t-SNE Projection', width=800, height=600)
    fig_2d.show()

    # 3D plot
    tsne_3d = TSNE(n_components=3, random_state=42)
    reduced_vectors_3d = tsne_3d.fit_transform(vectors)

    fig_3d = go.Figure()
    for doc_type in unique_types:
        idx = doc_types == doc_type
        fig_3d.add_trace(go.Scatter3d(
            x=reduced_vectors_3d[idx, 0], 
            y=reduced_vectors_3d[idx, 1],
            z=reduced_vectors_3d[idx, 2],
            mode='markers', 
            name=doc_type,
            text=headlines[idx],
            hoverinfo='text+name',
            marker=dict(size=5, opacity=0.8)
    ))
    fig_3d.update_layout(
        title='Semantic Clustering: 3D t-SNE Projection', 
        width=900, height=700,
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig_3d.show()

else:
    print("Vector database is still empty.")


Generating 2D and 3D t-SNE Plot...


### 4. Two-Stage Production Retrieval: Bi-Encoder + Cross-Encoder
A standard vector search (Bi-Encoder) is fast but can miss nuanced context. We implement a Two-Stage RAG architecture:
1. **Stage 1 (Recall):** Bi-encoder (`bge-large`) retrieves the top 20 candidates.
2. **Stage 2 (Precision):** Local Cross-Encoder (`ms-marco`) reranks candidates based on deep query-document attention, returning the top 5.

In [8]:
# Initialize local Cross-Encoder for Reranking (Replaces expensive LLM API calls)
print("\nLoading BAAI/bge-reranker-base...")
cross_encoder = CrossEncoder('BAAI/bge-reranker-base') #bge-reranker-large is even better

def fetch_and_rerank(query, top_k_retrieve=20, top_k_return=5): 
    """Two-stage retrieval pipeline."""
    # Stage 1: Fast Vector Search
    query_vector = encoder.encode(query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=top_k_retrieve)
    
    if not results['documents'][0]:
        return[]

    retrieved_docs = results['documents'][0]

    # Stage 2: Cross-Encoder Reranking
    # Create pairs of (Query, Document) for the cross-encoder
    cross_inp = [[query, doc] for doc in retrieved_docs]
    scores = cross_encoder.predict(cross_inp)

    # Sort documents by cross-encoder score
    ranked_indices = np.argsort(scores)[::-1]

    return [retrieved_docs[i] for i in ranked_indices[:top_k_return]]


Loading BAAI/bge-reranker-base...


In [9]:
# Test the Pipeline
test_query = "Who won the IIOTY award?"
print(f"\nTesting Pipeline with query: '{test_query}'")
top_docs = fetch_and_rerank(test_query)
if top_docs:
    print(f"Top Reranked Context:\n{top_docs[0]}")
else:
    print("No docs found.")


Testing Pipeline with query: 'Who won the IIOTY award?'
Top Reranked Context:
## Other HR Notes
- Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.  
- She was recognized for her contributions with the prestigious Insurellm IIOTY Innovator Award in 2023.  
- Maxine is currently involved in the women-in-tech initiative and participates in mentorship programs to guide junior employees.  
- Future development areas include improving her stakeholder communication skills to ensure smoother project transitions and collaboration.


### Comparing Pipeline

Phase 1: Fast Vector Search (Bi-Encoder)
Phase 2: Cross-Encoder Reranking

In [10]:
def compare_pipeline(query, top_k_retrieve=5, top_k_return=2):
    print(f"\n{'='*60}")
    print(f"🔍 QUERY: '{query}'")
    print(f"{'='*60}")
    
    # STAGE 1: Fast Vector Search (Bi-Encoder)
    query_vector = encoder.encode(query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=top_k_retrieve)
    
    if not results['documents'] or not results['documents'][0]:
        print("No documents found in database.")
        return
        
    retrieved_docs = results['documents'][0]
    
    print("\n❌ BEFORE RERANKING (Stage 1: Vector Search Only)")
    print("Often prioritizes keyword overlap over actual meaning. (Rank 1 might be wrong!)")
    for i in range(min(top_k_return, len(retrieved_docs))):
        # Truncating to 200 characters so it's easy to read in the console
        snippet = retrieved_docs[i].replace('\n', ' ')[:200]
        print(f"\n[Rank {i+1}] {snippet}...")
        
    # STAGE 2: Cross-Encoder Reranking
    cross_inp = [[query, doc] for doc in retrieved_docs]
    scores = cross_encoder.predict(cross_inp)
    
    # Sort documents by cross-encoder score
    ranked_indices = np.argsort(scores)[::-1]
    reranked_docs = [retrieved_docs[i] for i in ranked_indices]
    
    print("\n\n✅ AFTER RERANKING (Stage 2: Cross-Encoder)")
    print("Understands logic and context. The correct answer is pushed to Rank 1.")
    for i in range(min(top_k_return, len(reranked_docs))):
        snippet = reranked_docs[i].replace('\n', ' ')[:200]
        print(f"\n[Rank {i+1}] {snippet}...")

# Test it out!
test_query = "Who won the IIOTY award?"
compare_pipeline(test_query)

# Try a tricky negative query to see the reranker really shine
tricky_query = "When was Insurellm founded?"
compare_pipeline(tricky_query)


🔍 QUERY: 'Who won the IIOTY award?'

❌ BEFORE RERANKING (Stage 1: Vector Search Only)
Often prioritizes keyword overlap over actual meaning. (Rank 1 might be wrong!)

[Rank 1] ## Other HR Notes - Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.   - She was recognized for her contributions with the prestigi...

[Rank 2] - **2020:** Rating: 4.4/5   *Strong performance during challenging year. Successfully pivoted technology strategy for remote-first world.*  - **2019:** Rating: 4.8/5   *Exceptional leadership. Scaled ...


✅ AFTER RERANKING (Stage 2: Cross-Encoder)
Understands logic and context. The correct answer is pushed to Rank 1.

[Rank 1] ## Other HR Notes - Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.   - She was recognized for her contributions with the prestigi...

[Rank 2] ## Annual Performance History - **2023:** Rating: 4.9/5   *Except

### 5. System Evaluation
To ensure production readiness, we evaluate the system quantitatively against a ground-truth dataset using modular metrics (MRR, nDCG, and Keyword Coverage).

*(Metrics engine isolated in `scripts/custom_eval.py`)*

In [11]:
import sys
import json
import time
from pathlib import Path 
from tqdm.auto import tqdm

# Get the directory of the current notebook and resolve the parent root folder (01_rag_foundations)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Add project root to sys.path so 'scripts' is recognized as a package/module folder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Add scripts directory to path
sys.path.append(str(Path("scripts").resolve()))
from scripts.custom_eval import (
    calculate_precision_at_k,
    calculate_recall_at_k,
    calculate_mrr,
    calculate_ndcg,
    calculate_keyword_coverage,
    evaluate_rag_triad_with_llm
)

# Configuration Paths
TESTS_FILE = PROJECT_ROOT / "data" / "tests.jsonl"
EVAL_CACHE_FILE = "eval_results_cache.json"
MAX_EVAL_QUERIES = 5 # 12 Keeps total API calls under 25-30 requests per run

# ...................... Load test.jsonl dataset ........................................
test_dataset = []
if TESTS_FILE.exists():
    with open(TESTS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                test_dataset.append(json.loads(line))
    print(f"✅ Successfully loaded {len(test_dataset)} test queries from {TESTS_FILE.name}")
else:
    print(f"❌ Error: Could not find {TESTS_FILE}. Please check the file path.")

✅ Successfully loaded 150 test queries from tests.jsonl


In [12]:
# ...................... Offline Retrieval BenchMarks(No API calls)......................
print("\n")
print("📊 RETRIEVAL & RANKING METRICS (Offline Benchmarks)")
print("=" * 60)

all_p_at_3 = []
all_r_at_3 = []
first_hit_ranks = []
all_ndcg = []

for case in tqdm(test_dataset, desc="Running Local Retrieval Eval"):
    query = case["question"]
    keywords = case["keywords"]
    
    # 1. Fetch chunks using local Bi-Encoder + Cross-Encoder
    top_docs = fetch_and_rerank(query, top_k_retrieve=10, top_k_return=5)
    
    # 2. Determine binary relevance based on keyword presence
    rel_scores = []
    for doc in top_docs:
        doc_lower = doc.lower()
        # Count how many of the ground-truth keywords appear in the retrieved chunk
        found = sum(1 for kw in keywords if kw.lower() in doc_lower)
        
        # Heuristic: If at least 50% of the keywords are present, count it as a "Hit"
        is_relevant = 1 if (len(keywords) > 0 and (found / len(keywords)) >= 0.5) else 0
        rel_scores.append(is_relevant)
        
    # Pad to 5 elements if fetch_and_rerank returned fewer chunks
    while len(rel_scores) < 5:
        rel_scores.append(0)
        
    # 3. Compute Metrics for this specific query
    all_p_at_3.append(calculate_precision_at_k(rel_scores, k=3))
    
    # Assume 1 perfect document exists per query for Recall baseline
    all_r_at_3.append(calculate_recall_at_k(rel_scores, total_relevant=1, k=3))
    
    # Track MRR (Index of the first '1' in our list)
    try:
        first_hit = rel_scores.index(1) + 1
    except ValueError:
        first_hit = 0
    first_hit_ranks.append(first_hit)
    
    all_ndcg.append(calculate_ndcg(rel_scores, k=5))

# Print aggregate results
print(f"• Mean Precision@3: {np.mean(all_p_at_3):.4f}")
print(f"• Mean Hit Rate@3:  {np.mean(all_r_at_3):.4f} (Proxy for Recall)")
print(f"• MRR:              {calculate_mrr(first_hit_ranks):.4f}")
print(f"• Mean nDCG@5:      {np.mean(all_ndcg):.4f}")



📊 RETRIEVAL & RANKING METRICS (Offline Benchmarks)


Running Local Retrieval Eval:   0%|          | 0/150 [00:00<?, ?it/s]

• Mean Precision@3: 0.6022
• Mean Hit Rate@3:  1.8067 (Proxy for Recall)
• MRR:              0.9180
• Mean nDCG@5:      0.9065


In [13]:
# ................... LLM-AS-A-JUDGE (API Capped Subset) ...............................

# Select only the first N queries to protect API limits
golden_test_cases = test_dataset[:MAX_EVAL_QUERIES]

# Load existing evaluations from disk if available
eval_results = []
if Path(EVAL_CACHE_FILE).exists():
    with open(EVAL_CACHE_FILE, "r", encoding="utf-8") as f:
        eval_results = json.load(f)
    print(f"\nLoaded {len(eval_results)} pre-evaluated cases from cache.")

evaluated_queries = {res["query"] for res in eval_results}

print("\n")
print(f"🤖 RUNNING LLM-AS-A-JUDGE (Max {len(golden_test_cases)} cases)")
print("=" * 60)

for case in tqdm(golden_test_cases, desc="Evaluating Generation"):
    query = case["question"]
    keywords = case["keywords"]
    ground_truth = case["reference_answer"]
    
    # Skip if already evaluated and cached
    if query in evaluated_queries:
        continue

    # 1. Local Retrieval (0 API calls)
    t0 = time.time()
    top_docs = fetch_and_rerank(query, top_k_retrieve=10, top_k_return=2)
    context_str = "\n---\n".join(top_docs) if top_docs else "No context retrieved."
    retrieval_time = time.time() - t0
    print(f"\n[Stats] Retrieval: {retrieval_time:.2f}s | Query: {query[:30]}...")

    # 2. Generation Call (API Call #1)
    t1 = time.time()
    try: 
        generation_prompt = f"Answer using only the context:\n\n{context_str}\n\nQuery: {query}"
        gen_response = gemini.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": generation_prompt}],
            timeout=25.0
        )
        generated_answer = gen_response.choices[0].message.content
    except Exception as e:
        print(f"❌ API Generation Failed (Check limits): {e}")
        break
    gen_time = time.time() - t1
    print(f"[Stats] Gemini Generation: {gen_time:.2f}s")
    
    # 4.5-second pause to prevent hitting the 15 RPM limit
    time.sleep(4.5)

    # 3. Lexical Keyword Coverage (0 API calls)
    coverage = calculate_keyword_coverage(generated_answer, keywords)

    # 4. LLM Judge Call (API Call #2)
    t2 = time.time()
    try:
        verdict = evaluate_rag_triad_with_llm(
            client=gemini,
            model=MODEL,
            query=query,
            retrieved_context=context_str,
            generated_answer=generated_answer,
            ground_truth=ground_truth,
            timeout=25.0,
            rate_limit_delay=4.5 # Extra 4.5s delay before judge prompt
        )
    except Exception as e:
        print(f"❌ API Judge Failed (Retries exhausted): {e}")
        break
    judge_time = time.time() - t2
    print(f"[Stats] Gemini Judge: {judge_time:.2f}s | Passed: {verdict.passed}")

    # Save Results
    eval_results.append({
        "query": query,
        "generated_answer": generated_answer,
        "keyword_coverage": coverage,
        "faithfulness": verdict.faithfulness_score,
        "relevancy": verdict.relevancy_score,
        "passed": verdict.passed,
        "critique": verdict.faithfulness_critique
    })

    # Save to disk after every query to prevent data loss if interrupted
    with open(EVAL_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2)


Loaded 5 pre-evaluated cases from cache.


🤖 RUNNING LLM-AS-A-JUDGE (Max 5 cases)


Evaluating Generation:   0%|          | 0/5 [00:00<?, ?it/s]

In [14]:
# ......................Summary Dashboard................................
# Display summary results
print("\n")
print("📈 GENERATION SUMMARY REPORT")
print("=" * 60)
if eval_results:
    avg_faithfulness = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
    avg_relevancy = sum(r["relevancy"] for r in eval_results) / len(eval_results)
    pass_rate = sum(1 for r in eval_results if r["passed"]) / len(eval_results) * 100

    print(f"Total Golden Queries Evaluated: {len(eval_results)}")
    print(f"Average Faithfulness:           {avg_faithfulness:.2f} / 5.0")
    print(f"Average Relevancy:              {avg_relevancy:.2f} / 5.0")
    print(f"Overall Pass Rate:              {pass_rate:.1f}%")



📈 GENERATION SUMMARY REPORT
Total Golden Queries Evaluated: 5
Average Faithfulness:           5.00 / 5.0
Average Relevancy:              5.00 / 5.0
Overall Pass Rate:              100.0%
